# Hough Transform for Straight Line Detection

This notebook demonstrates **line detection** in images using the **Hough Transform**. The Hough Transform maps edge points from the image space into a parameter space where lines are represented as points of intersection, enabling robust detection of straight lines even in the presence of noise, occlusion, and gaps.

The analysis covers:

1. **Standard Hough Line Transform** (`cv2.HoughLines`) — detects lines by accumulating votes in HoughSpace.
2. **Probabilistic Hough Line Transform** (`cv2.HoughLinesP`) — a more efficient variant that directly returns line segment endpoints.
3. **Parameter Sensitivity** — how changes to `rho`, `theta`, threshold, `minLineLength`, and `maxLineGap` affect detection results.

All techniques are applied to `BMW.jpeg` and the results are visualized alongside the original image and edge map for direct comparison.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
image_bgr = cv2.imread('BMW.jpeg')

if image_bgr is None:
    print('Error: Could not load the image. Please check the file path.')
else:
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    image_gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(image_gray, (5, 5), 0)
    edges = cv2.Canny(blurred, 50, 150)

    print(f'Image loaded successfully. Shape: {image_rgb.shape}')
    print(f'Edge pixels: {np.sum(edges > 0)}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(image_rgb)
    axes[0].set_title('Original Image')
    axes[0].axis('off')

    axes[1].imshow(edges, cmap='gray')
    axes[1].set_title('Canny Edge Map')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

## Standard Hough Transform

The standard Hough Transform (`cv2.HoughLines`) detects lines by accumulating votes in HoughSpace. Each edge point casts a vote for every $(\rho, \theta)$ pair that satisfies $\rho = x\cos\theta + y\sin\theta$. Cells with votes above the threshold are identified as detected lines.

In [ ]:
lines = cv2.HoughLines(edges, rho=1, theta=np.pi/180, threshold=100)

h_display = image_rgb.copy()
if lines is not None:
    for rho, theta in lines[:, 0, :]:
        a = np.cos(theta)
        b = np.sin(theta)
        x0 = a * rho
        y0 = b * rho
        x1 = int(x0 + 1000 * (-b))
        y1 = int(y0 + 1000 * a)
        x2 = int(x0 - 1000 * (-b))
        y2 = int(y0 - 1000 * a)
        cv2.line(h_display, (x1, y1), (x2, y2), (0, 0, 255), 2)

plt.figure(figsize=(12, 8))
plt.imshow(h_display)
plt.title(f'Standard Hough Transform (threshold=100, {len(lines)} lines detected)')
plt.axis('off')
plt.show()

## Probabilistic Hough Transform

The Probabilistic Hough Transform (`cv2.HoughLinesP`) is a more efficient variant that returns line segment endpoints directly. It samples a subset of edge points and extends lines only within a region of interest, making it faster and more suitable for detecting line segments with gaps.

In [ ]:
lines_p = cv2.HoughLinesP(edges, rho=1, theta=np.pi/180, threshold=50,
                                  minLineLength=50, maxLineGap=20)

p_display = image_rgb.copy()
if lines_p is not None:
    for x1, y1, x2, y2 in lines_p[:, 0, :]:
        cv2.line(p_display, (x1, y1), (x2, y2), (0, 255, 0), 2)

plt.figure(figsize=(12, 8))
plt.imshow(p_display)
plt.title(f'Probabilistic Hough Transform (threshold=50, {len(lines_p)} segments detected)')
plt.axis('off')
plt.show()

## Parameter Sensitivity — Varying Threshold

The detection threshold controls the minimum number of votes required to identify a line. A lower threshold detects more lines (including noise), while a higher threshold only detects strong, well-supported lines.

In [ ]:
thresholds = [30, 80, 150, 200]
fig, axes = plt.subplots(1, len(thresholds) + 1, figsize=(24, 5))

axes[0].imshow(edges, cmap='gray')
axes[0].set_title('Edge Map')
axes[0].axis('off')

for i, t in enumerate(thresholds):
    lines_t = cv2.HoughLines(edges, rho=1, theta=np.pi/180, threshold=t)
    display = image_rgb.copy()
    if lines_t is not None:
        for rho, theta in lines_t[:, 0, :]:
            a = np.cos(theta)
            b = np.sin(theta)
            x0 = a * rho
            y0 = b * rho
            x1 = int(x0 + 1000 * (-b))
            y1 = int(y0 + 1000 * a)
            x2 = int(x0 - 1000 * (-b))
            y2 = int(y0 - 1000 * a)
            cv2.line(display, (x1, y1), (x2, y2), (0, 0, 255), 2)
    axes[i + 1].imshow(display)
    n_lines = len(lines_t) if lines_t is not None else 0
    axes[i + 1].set_title(f'Threshold = {t}\nLines: {n_lines}')
    axes[i + 1].axis('off')

plt.tight_layout()
plt.show()

## Parameter Sensitivity — Varying `rho` and `theta` Resolution

The `rho` and `theta` parameters control the angular resolution of the Hough accumulator. Coarser resolution is faster but less precise; finer resolution is slower but more accurate.

In [ ]:
configs = [
    (1, np.pi/180, 'rho=1, theta=1 deg'),
    (2, np.pi/90, 'rho=2, theta=2 deg'),
    (5, np.pi/36, 'rho=5, theta=5 deg'),
]

fig, axes = plt.subplots(1, len(configs) + 1, figsize=(24, 5))

axes[0].imshow(edges, cmap='gray')
axes[0].set_title('Edge Map')
axes[0].axis('off')

for i, (rho, theta, label) in enumerate(configs):
    lines_r = cv2.HoughLines(edges, rho=rho, theta=theta, threshold=100)
    display = image_rgb.copy()
    if lines_r is not None:
        for r, t in lines_r[:, 0, :]:
            a, b = np.cos(t), np.sin(t)
            x0, y0 = a * r, b * r
            x1 = int(x0 + 1000 * (-b))
            y1 = int(y0 + 1000 * a)
            x2 = int(x0 - 1000 * (-b))
            y2 = int(y0 - 1000 * a)
            cv2.line(display, (x1, y1), (x2, y2), (0, 0, 255), 2)
    axes[i + 1].imshow(display)
    n_lines = len(lines_r) if lines_r is not None else 0
    axes[i + 1].set_title(f'{label}\nLines: {n_lines}')
    axes[i + 1].axis('off')

plt.tight_layout()
plt.show()

## Parameter Sensitivity — Probabilistic Hough: `minLineLength` and `maxLineGap`

`minLineLength` filters out short line segments. `maxLineGap` bridges gaps between collinear segments. Together they control the granularity and connectivity of detected lines.

In [ ]:
configs_p = [
    (50, 10, 'minLen=50, maxGap=10'),
    (100, 30, 'minLen=100, maxGap=30'),
    (150, 60, 'minLen=150, maxGap=60'),
]

fig, axes = plt.subplots(1, len(configs_p) + 1, figsize=(24, 5))

axes[0].imshow(edges, cmap='gray')
axes[0].set_title('Edge Map')
axes[0].axis('off')

for i, (ml, mg, label) in enumerate(configs_p):
    lines_p2 = cv2.HoughLinesP(edges, rho=1, theta=np.pi/180,
                                   threshold=50, minLineLength=ml, maxLineGap=mg)
    display = image_rgb.copy()
    if lines_p2 is not None:
        for x1, y1, x2, y2 in lines_p2[:, 0, :]:
            cv2.line(display, (x1, y1), (x2, y2), (0, 255, 0), 2)
    axes[i + 1].imshow(display)
    n_seg = len(lines_p2) if lines_p2 is not None else 0
    axes[i + 1].set_title(f'{label}\nSegments: {n_seg}')
    axes[i + 1].axis('off')

plt.tight_layout()
plt.show()

## Combined Comparison

A 2x3 grid displaying the original image, edge map, standard Hough result, probabilistic Hough result, and parameter sensitivity views for comprehensive comparison.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

axes[0, 0].imshow(image_rgb)
axes[0, 0].set_title('Original Image')
axes[0, 0].axis('off')

axes[0, 1].imshow(edges, cmap='gray')
axes[0, 1].set_title('Canny Edges')
axes[0, 1].axis('off')

h_display = image_rgb.copy()
if lines is not None:
    for rho, theta in lines[:, 0, :]:
        a, b = np.cos(theta), np.sin(theta)
        x0, y0 = a * rho, b * rho
        x1 = int(x0 + 1000 * (-b))
        y1 = int(y0 + 1000 * a)
        x2 = int(x0 - 1000 * (-b))
        y2 = int(y0 - 1000 * a)
        cv2.line(h_display, (x1, y1), (x2, y2), (0, 0, 255), 2)
axes[0, 2].imshow(h_display)
n_h = len(lines) if lines is not None else 0
axes[0, 2].set_title(f'Standard Hough (threshold=100)\nLines: {n_h}')
axes[0, 2].axis('off')

p_display = image_rgb.copy()
if lines_p is not None:
    for x1, y1, x2, y2 in lines_p[:, 0, :]:
        cv2.line(p_display, (x1, y1), (x2, y2), (0, 255, 0), 2)
axes[1, 0].imshow(p_display)
n_p = len(lines_p) if lines_p is not None else 0
axes[1, 0].set_title(f'Probabilistic Hough\nSegments: {n_p}')
axes[1, 0].axis('off')

axes[1, 1].imshow(edges, cmap='gray')
axes[1, 1].set_title('Canny Edges (Hough Input)')
axes[1, 1].axis('off')

axes[1, 2].axis('off')
axes[1, 2].set_title('Parameter sensitivity\nsee cells above')

plt.tight_layout()
plt.show()